# Galaxien: oktonische Orientierung und Spin-Bias (SageMath)

Dieses Notebook nutzt **SageMath** (Oktaven-Algebra). In Jupyter / VS Code: **Kernel auf SageMath bzw. „Sage“ stellen** (nicht den reinen Python-3-Kernel), sonst schlagen `Octonions` u. a. fehl.

Optional lokal: `sage -n jupyter` oder Sage aus dem Installations-Startmenü mit Jupyter-Notebook.

## 1. Oktaven über $\mathbb{Q}$ (Fano-Ebene / Cayley-Struktur in Sage)

In [ ]:
O = Octonions(QQ)
B = list(O.basis())
print("Dimension (als Q-Vektorraum):", O.degree())
print("Anzahl Basiselemente (inkl. 1):", len(B))
print("1 = B[0]:", B[0])
# Eine der sieben (imaginären) Fano-Richtungen; Quadrat in Cayley-Algebra:
e1 = B[1]
print("e1^2 =", e1 * e1)

## 2. Spin-Simulation (±1) mit leichtem Bias – motiviert als „Richtung“ relativ zur oktonischen Wahl

Platzhalter-Modell: viele unabhängige Zufalls-Entscheidungen mit verschobener 0,5-Schwelle. Optional nutzen wir Sages `set_random_seed` für Reproduzierbarkeit.

In [ ]:
import random as _py_random


def check_octonionic_bias(n_galaxies, bias_factor=0.01, seed=None):
    """Mittelwert der ±1-Spins; bias_factor hebt p(+1) leicht an."""
    if seed is not None:
        set_random_seed(seed)
        _py_random.seed(seed)
    if n_galaxies <= 0:
        return float("nan")
    s = 0.0
    for _ in range(n_galaxies):
        u = _py_random.random()
        spin = 1 if (u + bias_factor) > 0.5 else -1
        s += spin
    return s / n_galaxies

## 3. Lauf (SDSS-ähnliche Größenordnung dauert nur Sekundenbruchteile)

Bei `n=200_000` siehst du die erwartet kleine Abweichung des Mittelwerts von 0.

In [ ]:
n_gal, bias = 200_000, 0.01
seed = 1
m = check_octonionic_bias(n_gal, bias_factor=bias, seed=seed)
print(f"n = {n_gal}, bias_factor = {bias}, seed = {seed}")
print(f"Mittlerer Bias (Durchschnitt ±1): {float(m):.6f}")

## 4. Kurz prüfen: gleiche Werte bei zweitem Aufruf mit gleichem `seed`

In [ ]:
m2 = check_octonionic_bias(10_000, 0.01, seed=42)
m2b = check_octonionic_bias(10_000, 0.01, seed=42)
# Float-Vergleich: identisch, wenn derselbe Zufallspfad
ok = m2 == m2b
print("Reproduzierbar (exakt):", ok, "| m2, m2b =", m2, m2b)

## 5. Erweiterte Simulation: Oktave, lokaler $e_1$–$e_3$-Anteil, globaler $e_7$-Drall

Skript im Projekt: `galaxien_octonionic_spin.sage` (Start: `sage galaxien_octonionic_spin.sage`).

Wahrscheinlichkeit für „cw“: $p \\approx 0{,}5 + |\\texttt{drift\\_scale}|$ (Orthonormalbasis: $|e_7|=1$).

In [ ]:
import random


def octonionic_spin_simulation(
    n_galaxies=200_000,
    drift_scale=None,
    seed=None,
):
    if drift_scale is None:
        drift_scale = QQ(1) / 200
    if seed is not None:
        set_random_seed(int(seed))
        random.seed(int(seed))
    if n_galaxies <= 0:
        return float("nan")
    O = Octonions(QQ)
    basis = list(O.basis())
    e1, e2, e3 = basis[1], basis[2], basis[3]
    global_drift = basis[7] * drift_scale
    nd = abs(RR(drift_scale))
    p_cw = min(RR(0.999), max(RR(0.001), RR(0.5) + nd))
    count_cw = count_ccw = 0
    for _ in range(n_galaxies):
        local_v = (
            random.uniform(-1, 1) * e1
            + random.uniform(-1, 1) * e2
            + random.uniform(-1, 1) * e3
        )
        _ = local_v + global_drift
        if random.random() < p_cw:
            count_cw += 1
        else:
            count_ccw += 1
    return float((count_cw - count_ccw) / n_galaxies)


n = 200_000
result_bias = octonionic_spin_simulation(n, drift_scale=QQ(5) / 1000, seed=1)
print(f"Simulation mit n={n}")
print(f"Okt. induzierter Bias: {result_bias:.5f}")